# Merging NOAA, egalei, and ERA5

The goal of this notebook is to merge the NOAA (as a time series) and eaglei data using FIPS, datetime, and latitude/longitude (of the county centroid) as a multi-index

Then it will merge this with corresponding ERA5 data

## Imports

In [2]:
import pandas as pd
import dask.dataframe as dd
import xarray as xr

# Indexing NOAA and eaglei by time

The eaglei data wasn't saved with a timeseries index, and didn't have FIPS as part of its index
This code will covert the original exported eaglei data into the right type of time series

In [2]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet file
eaglei = pd.read_parquet('../Data/Merged_Data/eaglei_outages_with_county_info.parquet')

#Make datetime and FIPS a multiindex for eaglei
eaglei['time'] = pd.to_datetime(eaglei['datetime'])
eaglei.set_index(['time', 'FIPS'], inplace=True)

#Export eaglei as a parquet file
eaglei.to_parquet('../Data/Merged_Data/eaglei_outages_with_county_info_timeseries.parquet')

# Merging NOAA and eaglei

The datasets we want to work with are pretty large, so we'll load them as dask dataframes

In [3]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet file
eaglei = dd.read_parquet('../Data/Merged_Data/eaglei_outages_with_county_info_timeseries.parquet')

#Load the ../Data/NOAA_Cleaned_ExplodedFIPS.parquet file
noaa = dd.read_parquet('../Data/NOAA_Cleaned_Data/NOAA_Timeseries.parquet')

#Merge eaglei and noaa based on their indices
eaglei_noaa = eaglei.merge(noaa, left_index=True, right_index=True, how='left')

#Export eaglei_noaa to a parquet file
eaglei_noaa.to_parquet('../Data/Merged_Data/eaglei_noaa.parquet', engine='pyarrow')

If we want to work with already-exported merged data, we can just load the file:

In [ ]:
#Load the eaglei_noaa merged data
eaglei_noaa = pd.read_parquet('../Data/Merged_Data/eaglei_noaa.parquet')

# Adding Geospatial Indices to eaglei-NOAA

To merge with ERA5 data, the eaglei-NOAA data will need to have latitude and longitude as indices/coordinates.

In [6]:
#Load the eaglei_noaa dataframe
eaglei_noaa = pd.read_parquet('../Data/Merged_Data/eaglei_noaa.parquet')

In [7]:
#Use fips_code, datetime, centroid_latitude and centroid_longitude as indices.
#Since dask doesn't support multiindexes, we'll keep these as variables as well
eaglei_noaa.set_index(['fips_code', 'datetime', 'centroid_latitude', 'centroid_longitude'], inplace=True, drop=False)

#Rename the indices FIPS, time, latitude, and longitude
eaglei_noaa.index.names = ['FIPS', 'time', 'latitude', 'longitude']

#Export eaglei_noaa as a parquet file eaglei_noaa_latlon
eaglei_noaa.to_parquet('../Data/Merged_Data/eaglei_noaa_latlon.parquet')

# Converting ERA5 to parquet

To perform the merging of ERA5 and the NOAA-eaglei data, we'll need both to be in parquet (or, at least, non-grib) format

In [ ]:
# Start by converting the ERA5 2016 grib file to parquet

# Load the GRIB file ../Data/ERA5_2018.grib using the cfgrib engine and setting decode_timedelta to be true
grib_data = xr.open_dataset('../Data/ERA5_Data/ERA5_2015.grib', engine='cfgrib', decode_timedelta=True)

# Convert to an xarray DataFrame
df = grib_data.to_dataframe()

# Save as Parquet file
df.to_parquet('../Data/ERA5_Data/ERA5_2015.parquet')

# Merging NOAA-eaglei and ERA5

We can use pyarrow or dask to merge the parquet files

In [11]:
# Read Parquet files into Dask DataFrames
df1 = dd.read_parquet('../Data/Merged_Data/eaglei_noaa_latlon.parquet', engine='pyarrow')
df2 = dd.read_parquet('../Data/ERA5_Data/ERA5_2018.parquet', engine='pyarrow')



In [3]:
era = pd.read_parquet('../Data/ERA5_Data/ERA5_2014.parquet')

In [9]:
#Count the number of non-nan in each variable of era
era.count()


number        225823464
surface       225823464
valid_time    225823464
t2m           156992340
u10           156992340
v10           156992340
sf            156992340
tp            156992340
dtype: int64

In [12]:
#Get a list of columns in df1
df1.columns

Index(['fips_code', 'customers_out', 'datetime', 'YEAR', 'NAME', 'STUSPS',
       'Pct_Buried_Lines', 'neighbors', 'Subregion', 'centroid_longitude',
       'centroid_latitude', 'centroid_rounded', 'POPULATION', 'BUILDVALUE',
       'AGRIVALUE', 'AREA', 'SOVI_SCORE', 'Power_Dependent_Devices_DME_Mean',
       'event_count SnowIce', 'event_count Flood', 'event_count Storm',
       'event_count Hurricane', 'event_count Heat', 'event_count Fire',
       'event_count Wind', 'event_count Ocean', 'event_count Other'],
      dtype='object')

In [13]:
df2.columns

Index(['number', 'surface', 'valid_time', 't2m', 'u10', 'v10', 'sf', 'tp'], dtype='object')

In [14]:
# Merge the DataFrames
result = df1.merge(df2, how='left', left_index=True, right_index=True)

In [19]:
#List the index values in result
result.info()

<class 'dask.dataframe.dask_expr.DataFrame'>
Columns: 35 entries, fips_code to tp
dtypes: datetime64[ns](2), float32(5), float64(21), int32(1), int64(1), string(5)

In [22]:
#Write a name_function that returns a string that includes one component for each level in the index of result
#def name_function(row):
#    return f"{row['fips_code']}_{row['time'].strftime('%Y-%m-%d')}_{row['centroid_latitude']}_{row['centroid_longitude']}"


#result.to_parquet('../Data/Merged_Data/ERA5Merge', name_function=name_function)

#Use dask to export result as a parquet file
result = result.reset_index()
#result.to_parquet('../Data/Merged_Data/ERA5Merge.parquet')
result.to_parquet('../Data/Merged_Data/ERA5Merge.parquet', write_options={'compression': 'snappy'},partition_on=['fips_code'])

TypeError: 'name' must be a list / sequence of column names.